In [0]:
# =============================================================================
# NOTEBOOK  : bronze_pos_transactions_batch
# PURPOSE   : Ingest pos_transactions_sample.csv → bronze.pos_transactions
# LAYER     : Bronze (raw ingestion — no transformation)
# FREQUENCY : Daily at 2 AM — triggered by batch orchestrator
# FORMAT    : CSV
# NOTE      : Runs AFTER streaming pipeline is stopped by orchestrator
#             Same target table as stream notebook
#             _source = "pos_batch_csv" distinguishes from stream rows
#             After this completes, reconciliation notebook runs for that date
# =============================================================================
 
# ── CELL 1: Imports & Config ──────────────────────────────────────────────────
import sys, json
sys.path.insert(0, "/Workspace/Shared/mini_project_a3t7")
 
from utils.audit import start_audit, end_audit
from utils.quality_checks import assert_not_empty
from utils.schema_registry import BRONZE_POS_TRANSACTIONS_SCHEMA
 
from pyspark.sql.functions import current_timestamp, lit, col, to_date
 
with open("/Workspace/Shared/mini_project_a3t7/config/config.json") as f:
    cfg = json.load(f)
 
SOURCE_PATH  = cfg["landing_paths"]["pos_transactions_batch"]
TARGET_TABLE = cfg["tables"]["bronze_pos_transactions"]
CHECKPOINT   = cfg["checkpoint_paths"]["bronze_pos_transactions_batch"]
PIPELINE     = "bronze_pos_transactions_batch"

In [0]:
# ── Audit + Read + Write ─────────────────────────────────────────────
run_id = start_audit(spark, PIPELINE, TARGET_TABLE)
 
try:
    # ── READ ──────────────────────────────────────────────────────────────────
    df = (
        spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.schemaLocation", CHECKPOINT + "/schema")
        .schema(BRONZE_POS_TRANSACTIONS_SCHEMA)
        .load(SOURCE_PATH)
    )
 
    # ── ADD AUDIT COLUMNS ─────────────────────────────────────────────────────
    df = (
        df
        .withColumn("_source",         lit("pos_batch_csv"))
        .withColumn("source_file",    col("_metadata.file_path"))
        .withColumn("ingested_at",     current_timestamp())
        .withColumn("ingested_date",   to_date(current_timestamp()).cast("string"))
        .withColumn("pipeline_run_id", lit(run_id))
    )
 
    # ── WRITE ─────────────────────────────────────────────────────────────────
    query = (
        df.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", CHECKPOINT)
        .partitionBy("ingested_date")
        .trigger(availableNow=True)
        .toTable(TARGET_TABLE)
    )
 
    query.awaitTermination()
 
    # ── QUALITY CHECK ─────────────────────────────────────────────────────────
    written_df = (
        spark.read.table(TARGET_TABLE)
        .where(f"pipeline_run_id = '{run_id}'")
    )
    rows_written = written_df.count()
    assert_not_empty(written_df, PIPELINE)
 
    print(f"[WRITE] {rows_written} rows written to {TARGET_TABLE}")
 
    # ── END AUDIT (SUCCESS) ───────────────────────────────────────────────────
    end_audit(spark, run_id, "SUCCESS", rows_written=rows_written)
 
except Exception as e:
    end_audit(spark, run_id, "FAILED", error=str(e))
    raise